# Modulo 6 · Dashboards interactivos con Streamlit

**Streamlit** convierte un script de Python en una aplicacion web interactiva,
sin necesidad de saber HTML, CSS o JavaScript. Es la forma mas rapida de
compartir un analisis de BI con otras personas.

En este modulo entenderas los bloques basicos de Streamlit y luego vamos a
recorrer el dashboard final del curso: `app/dashboard.py`.

**Contenidos:**
1. Anatomia de una app de Streamlit
2. Widgets: filtros interactivos
3. Layout: columnas, sidebar, metricas
4. Cache de datos con `st.cache_data`
5. Recorrido del dashboard final `app/dashboard.py`


## 1. Anatomia de una app de Streamlit

Una app de Streamlit es un archivo `.py` normal. Cada vez que el usuario
interactua con un widget (un filtro, un boton), **el script completo se
vuelve a ejecutar de arriba a abajo**. Esto es clave para entender como
disenar tus apps.

Ejemplo minimo (guardado en `app/ejemplo_minimo.py` mas abajo lo crearemos):

```python
import streamlit as st
import pandas as pd

st.title("Mi primer dashboard")

df = pd.read_csv("data/ventas.csv")
region = st.selectbox("Elige una region", df["region"].unique())

df_filtrado = df[df["region"] == region]
st.metric("Ingreso total", f"${df_filtrado['ingreso'].sum():,.0f}")
st.dataframe(df_filtrado)
```

Para ejecutarla: `streamlit run app/ejemplo_minimo.py`


In [1]:
# Vamos a crear ese ejemplo minimo como un archivo real que puedas ejecutar
codigo_ejemplo = '''import streamlit as st
import pandas as pd

st.title("Mi primer dashboard")

df = pd.read_csv("data/ventas.csv")
region = st.selectbox("Elige una region", sorted(df["region"].unique()))

df_filtrado = df[df["region"] == region]
st.metric("Ingreso total", f"${df_filtrado[\'ingreso\'].sum():,.0f}")
st.dataframe(df_filtrado.head(20))
'''

with open("../app/ejemplo_minimo.py", "w") as f:
    f.write(codigo_ejemplo)

print("Creado app/ejemplo_minimo.py")
print("Ejecutalo desde la raiz del proyecto con: streamlit run app/ejemplo_minimo.py")


Creado app/ejemplo_minimo.py
Ejecutalo desde la raiz del proyecto con: streamlit run app/ejemplo_minimo.py


## 2. Widgets: filtros interactivos

Los widgets mas usados en un dashboard de BI:

| Widget | Uso tipico |
|---|---|
| `st.selectbox` | Elegir una unica opcion (ej: una region) |
| `st.multiselect` | Elegir varias opciones (ej: varias categorias) |
| `st.date_input` | Elegir un rango de fechas |
| `st.slider` | Elegir un rango numerico |
| `st.radio` | Elegir una opcion entre pocas alternativas |
| `st.button` | Disparar una accion (ej: recalcular, exportar) |

Cada widget retorna el valor seleccionado, que puedes usar directamente
para filtrar tu DataFrame, tal como viste arriba con `st.selectbox`.


## 3. Layout: columnas, sidebar y metricas

- `st.sidebar` ubica widgets en un panel lateral (ideal para filtros).
- `st.columns(n)` divide el espacio en `n` columnas, util para KPIs en fila.
- `st.metric(label, value)` muestra un numero grande estilo "tarjeta KPI".
- `st.tabs([...])` organiza contenido en pestanas.

```python
col1, col2, col3 = st.columns(3)
col1.metric("Ingreso total", "$12.500.000")
col2.metric("Utilidad total", "$4.200.000")
col3.metric("Margen", "33.6%")
```


## 4. Cache de datos con `st.cache_data`

Leer y limpiar un CSV en cada interaccion del usuario seria lento. El
decorador `@st.cache_data` guarda el resultado en memoria y solo vuelve a
ejecutar la funcion si cambian sus argumentos o el codigo:

```python
@st.cache_data
def cargar_datos():
    df = pd.read_csv("data/ventas.csv", parse_dates=["fecha"])
    # ... limpieza ...
    return df

df = cargar_datos()
```

Esto es exactamente lo que hace `app/dashboard.py`.


## 5. Recorrido del dashboard final: `app/dashboard.py`

Abre el archivo `app/dashboard.py` (en la raiz del proyecto) y sigue esta guia
mientras lo lees:

1. **`cargar_datos()`**: lee `data/ventas.csv`, aplica la limpieza del
   Modulo 3 (nulos, duplicados) y crea columnas derivadas, todo cacheado.
2. **Sidebar de filtros**: rango de fechas, region, categoria y segmento de
   cliente, usando `st.sidebar.date_input` y `st.sidebar.multiselect`.
3. **KPIs en columnas**: ingreso total, utilidad total, margen promedio,
   ticket promedio y numero de ventas, usando `st.columns` + `st.metric`.
4. **Tendencia mensual**: grafico de lineas con Plotly (ingreso vs utilidad).
5. **Top productos y participacion por categoria**: barras + torta, en dos
   columnas lado a lado.
6. **Heatmap region x categoria**: con `px.imshow`.
7. **Ranking de vendedores**: tabla con formato de moneda usando `st.dataframe`.
8. **Datos filtrados**: tabla completa dentro de un `st.expander` (colapsable).

Para ejecutarlo:

```bash
streamlit run app/dashboard.py
```

Se abrira en tu navegador en `http://localhost:8501`. Prueba cambiar los
filtros de la barra lateral y observa como se actualizan todos los graficos
y KPIs automaticamente.


## 🧠 Retos del modulo 6

1. Agrega al dashboard un nuevo filtro por `vendedor` (multiselect) en la sidebar.
2. Agrega una nueva metrica KPI: "unidades vendidas totales".
3. Agrega un `st.tabs` que separe la vista actual en dos pestanas:
   "Resumen ejecutivo" (KPIs + tendencia) y "Detalle" (el resto de graficos y tabla).
4. (Avanzado) Agrega un boton `st.download_button` que permita descargar
   `df_filtrado` como CSV.

No hay archivo de soluciones para este modulo: modifica directamente una copia
de `app/dashboard.py` y ejecutala con `streamlit run` para ver tus cambios.
